# Model CNN From Scratch

Notebook ini hanya mendefinisikan arsitektur CNN, menjalankan training, dan menyimpan model serta history. Evaluasi test set dan visualisasi evaluasi dipindahkan ke notebook evaluasi.

## 1. Arsitektur Custom CNN

Model dibuat dari layer Conv2D, BatchNormalization, ReLU, Dropout, GlobalAveragePooling, Dense, dan optional Squeeze-and-Excitation block tanpa pretrained weights.

In [ ]:
def make_optimizer(name: str, learning_rate: float):
    name = name.lower()
    if name == 'adamw' and hasattr(tf.keras.optimizers, 'AdamW'):
        return tf.keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=1e-4)
    return tf.keras.optimizers.Adam(learning_rate=learning_rate)


def conv_bn_relu(x: tf.Tensor, filters: int) -> tf.Tensor:
    x = tf.keras.layers.Conv2D(filters, kernel_size=3, padding='same', kernel_initializer='he_normal')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    return tf.keras.layers.ReLU()(x)


def squeeze_excite_block(x: tf.Tensor, ratio: int = 8) -> tf.Tensor:
    channels = int(x.shape[-1])
    se = tf.keras.layers.GlobalAveragePooling2D()(x)
    se = tf.keras.layers.Dense(max(channels // ratio, 8), activation='relu')(se)
    se = tf.keras.layers.Dense(channels, activation='sigmoid')(se)
    se = tf.keras.layers.Reshape((1, 1, channels))(se)
    return tf.keras.layers.Multiply()([x, se])


def build_custom_cnn(config: dict) -> tf.keras.Model:
    filters = config.get('filters', [32, 64, 128, 256])
    dropout_scale = float(config.get('dropout_scale', 1.0))
    use_se = bool(config.get('use_se', True))
    dense_units = int(config.get('dense_units', 128))
    optimizer_name = config.get('optimizer', 'adamw')
    learning_rate = float(config.get('learning_rate', 1e-3))

    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
    x = inputs
    block_dropouts = [0.15, 0.20, 0.25, 0.30]

    for block_idx, filter_count in enumerate(filters):
        x = conv_bn_relu(x, filter_count)
        x = conv_bn_relu(x, filter_count)
        if use_se and block_idx in [2, 3]:
            x = squeeze_excite_block(x)
        x = tf.keras.layers.MaxPooling2D()(x)
        x = tf.keras.layers.Dropout(block_dropouts[block_idx] * dropout_scale)(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(dense_units, activation='relu', kernel_initializer='he_normal')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.40 * dropout_scale)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs, outputs, name='custom_cnn_from_scratch')
    model.compile(
        optimizer=make_optimizer(optimizer_name, learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
    )
    return model


base_cnn_config = {
    'filters': [32, 64, 128, 256],
    'dropout_scale': 1.0,
    'use_se': True,
    'dense_units': 128,
    'optimizer': 'adamw',
    'learning_rate': 1e-3,
    'batch_size': 32,
}

build_custom_cnn(base_cnn_config).summary()

Model: "custom_cnn_from_scratch"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128, 128,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 128, 128,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 128,  │      9,248 │ re_lu[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 128, 128,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ re_lu_1[0][0]     │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64, 64,    │          0 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │     18,496 │ dropout[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 64, 64,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 64, 64,    │     36,928 │ re_lu_2[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 64, 64,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ re_lu_3[0][0]     │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32, 32,    │          0 │ max_pooling2d_1[

 Total params: 1,230,545 (4.69 MB)

 Trainable params: 1,228,369 (4.69 MB)

 Non-trainable params: 2,176 (8.50 KB)

## 2. Training CNN

Fungsi training mengembalikan model, checkpoint, history, dan konfigurasi. Prediksi test set dihitung nanti pada notebook evaluasi.

In [ ]:

def train_cnn_model(scenario_id: str, scenario_name: str, config: dict, augment_train: bool = False) -> tuple[dict, tf.keras.Model]:
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    batch_size = int(config.get('batch_size', BATCH_SIZE))
    train_ds = prepare_cnn_dataset(load_tf_split(scenario_name, 'train', shuffle=True, batch_size=batch_size), augment=augment_train)
    val_ds = prepare_cnn_dataset(load_tf_split(scenario_name, 'validation', shuffle=False, batch_size=batch_size))

    model = build_custom_cnn(config)
    checkpoint_path = OUTPUT_DIR / f'best_{scenario_id}.keras'
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            checkpoint_path,
            monitor='val_loss',
            save_best_only=True,
            mode='min',
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.4,
            patience=LR_PATIENCE,
            min_lr=1e-6,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=EARLY_STOP_PATIENCE,
            restore_best_weights=True,
        ),
    ]

    print(f'\nTraining {scenario_id}')
    print('Config:', config, 'augment_train:', augment_train)
    history_obj = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=MAX_EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    history = history_obj.history

    record = {
        'scenario': scenario_id,
        'scenario_name': scenario_name,
        'model_type': 'Custom CNN Conv2D from scratch',
        'config': {**config, 'augmentation': bool(augment_train), 'scenario_dataset': scenario_name},
        'history': {key: [float(v) for v in values] for key, values in history.items()},
        'checkpoint_path': str(checkpoint_path),
        'epochs_trained': len(history.get('loss', [])),
        'best_val_loss': float(min(history.get('val_loss', [np.inf]))),
        'best_val_accuracy': float(max(history.get('val_accuracy', [0.0]))),
    }
    return record, model


## 3. Skenario Training CNN

Skenario training mencakup plain/enhanced, crop, augmentation, SE ablation, dan tuning ringan. Semua hasil mentah disimpan untuk evaluasi berikutnya.

In [ ]:
MAX_EPOCHS = 50
print(f'CNN max epochs: {MAX_EPOCHS} | early stopping patience: {EARLY_STOP_PATIENCE}')

cnn_experiments = [
    {
        'scenario_id': 'C_cnn_plain_full_se_no_aug',
        'scenario_name': 'full_plain',
        'augment': False,
        'config': {**base_cnn_config, 'use_se': True},
        'purpose': 'CNN tanpa enhancement',
    },
    {
        'scenario_id': 'D_cnn_enhanced_full_se_no_aug',
        'scenario_name': 'full_enhanced',
        'augment': False,
        'config': {**base_cnn_config, 'use_se': True},
        'purpose': 'Pengaruh enhancement',
    },
    {
        'scenario_id': 'F_cnn_enhanced_full_se_aug',
        'scenario_name': 'full_enhanced',
        'augment': True,
        'config': {**base_cnn_config, 'use_se': True},
        'purpose': 'Pengaruh augmentation',
    },
]

if RUN_OPTIONAL_CROP_SCENARIO:
    cnn_experiments.append({
        'scenario_id': 'E_cnn_enhanced_crop_se_no_aug',
        'scenario_name': 'crop_enhanced',
        'augment': False,
        'config': {**base_cnn_config, 'use_se': True},
        'purpose': 'Pengaruh face crop',
    })

if RUN_SE_ABLATION:
    cnn_experiments.append({
        'scenario_id': 'G_cnn_enhanced_full_no_se_aug',
        'scenario_name': 'full_enhanced',
        'augment': True,
        'config': {**base_cnn_config, 'use_se': False},
        'purpose': 'Ablation SE block',
    })

if RUN_LIGHT_TUNING:
    cnn_experiments.extend([
        {
            'scenario_id': 'T1_cnn_enhanced_full_se_aug_lr5e4',
            'scenario_name': 'full_enhanced',
            'augment': True,
            'config': {**base_cnn_config, 'learning_rate': 5e-4, 'dropout_scale': 1.0, 'optimizer': 'adamw'},
            'purpose': 'Tuning learning rate',
        },
        {
            'scenario_id': 'T2_cnn_enhanced_full_se_aug_wider',
            'scenario_name': 'full_enhanced',
            'augment': True,
            'config': {**base_cnn_config, 'filters': [48, 96, 160, 256], 'learning_rate': 7e-4, 'dropout_scale': 1.1, 'optimizer': 'adamw'},
            'purpose': 'Tuning jumlah filter dan dropout',
        },
        {
            'scenario_id': 'T3_cnn_enhanced_full_se_aug_adam',
            'scenario_name': 'full_enhanced',
            'augment': True,
            'config': {**base_cnn_config, 'optimizer': 'adam', 'learning_rate': 1e-3, 'dropout_scale': 0.9, 'batch_size': 64},
            'purpose': 'Tuning optimizer dan batch size',
        },
    ])

trained_models = {}
cnn_training_records = {}
for experiment in cnn_experiments:
    record, model = train_cnn_model(
        experiment['scenario_id'],
        experiment['scenario_name'],
        experiment['config'],
        augment_train=experiment['augment'],
    )
    record['purpose'] = experiment['purpose']
    cnn_training_records[experiment['scenario_id']] = record
    trained_models[experiment['scenario_id']] = model
